# Exercise 5.5 — Length of the curve $c(t) = (2t+1,\; -t^2)$, $\;t \in [0, 1]$

**Derivative:** $\dot{c}(t) = (2,\; -2t)$

**Speed:** $\|\dot{c}(t)\| = \sqrt{4 + 4t^2} = 2\sqrt{1 + t^2}$

In [9]:
import numpy as np
from scipy import integrate

## Part 1 — Numerical length via Eq. 4.2

$$L(c) = \int_0^1 \|\dot{c}(t)\|\, dt = \int_0^1 2\sqrt{1+t^2}\, dt$$

We evaluate this with `scipy.integrate.quad`.

In [10]:
# --- Define the curve and its derivative ---
def c(t):
    """Curve c(t) = (2t+1, -t^2)."""
    return np.array([2*t + 1, -t**2])

def c_dot(t):
    """Derivative c'(t) = (2, -2t)."""
    return np.array([2.0, -2*t])

def speed(t):
    """||c'(t)|| = 2*sqrt(1 + t^2)."""
    return np.linalg.norm(c_dot(t))

# --- Part 1: Numerical integration (Eq. 4.2) ---
L_numerical, error = integrate.quad(speed, 0, 1)
print(f"Numerical length (quad):  L = {L_numerical:.10f}")
print(f"Integration error bound:      {error:.2e}")

Numerical length (quad):  L = 2.2955871494
Integration error bound:      2.55e-14


**Result:** `scipy.integrate.quad` computes $L \approx 2.2955871494$ with an integration error bound of $\approx 2.55 \times 10^{-14}$ — effectively machine precision. This confirms that adaptive quadrature is extremely accurate for smooth integrands like $2\sqrt{1+t^2}$.

## Part 2a — Analytical result

The integral has a closed form:

$$\int_0^1 2\sqrt{1+t^2}\,dt = \left[t\sqrt{1+t^2} + \ln\!\left(t + \sqrt{1+t^2}\right)\right]_0^1 = \sqrt{2} + \ln(1+\sqrt{2})$$

In [11]:
# --- Part 2a: Analytical length ---
L_analytical = np.sqrt(2) + np.log(1 + np.sqrt(2))
print(f"Analytical length:        L = {L_analytical:.10f}")
print(f"Absolute difference:          {abs(L_numerical - L_analytical):.2e}")
print(f"\n=> The numerical and analytical results agree.")

Analytical length:        L = 2.2955871494
Absolute difference:          0.00e+00

=> The numerical and analytical results agree.


**Result:** The closed-form expression $\sqrt{2} + \ln(1+\sqrt{2})$ agrees exactly with the numerical result — the absolute difference is $0.00 \times 10^0$. This validates both the analytical derivation and the quadrature implementation.

## Part 2b — Length via Eq. 4.5 (Riemann sum approximation)

Approximate the integral with a sum over $N$ subintervals:

$$L(c) \approx \sum_{i=0}^{N-1} \|\dot{c}(t_i)\|\,\Delta t, \qquad t_i = \frac{i}{N},\quad \Delta t = \frac{1}{N}$$

We use the analytic expression $\|\dot{c}(t)\| = 2\sqrt{1+t^2}$ directly.

In [12]:
# --- Part 2b: Riemann-sum approximation using analytic ||c'(t)|| ---
def speed_analytic(t):
    """Analytic expression: ||c'(t)|| = 2*sqrt(1 + t^2)."""
    return 2 * np.sqrt(1 + t**2)

print(f"{'N':>10}  {'L_approx':>14}  {'|error|':>12}")
print("-" * 42)

for N in [10, 100, 1_000, 10_000, 100_000]:
    dt = 1.0 / N
    t_vals = np.linspace(0, 1 - dt, N)          # left Riemann sum
    L_approx = np.sum(speed_analytic(t_vals)) * dt
    err = abs(L_approx - L_analytical)
    print(f"{N:>10}  {L_approx:>14.10f}  {err:>12.2e}")

print(f"\nAnalytical: {L_analytical:.10f}")

         N        L_approx       |error|
------------------------------------------
        10    2.2553444517      4.02e-02
       100    2.2914567989      4.13e-03
      1000    2.2951730537      4.14e-04
     10000    2.2955457292      4.14e-05
    100000    2.2955830073      4.14e-06

Analytical: 2.2955871494


**Result:** The left Riemann sum converges at $O(1/N)$ — each 10× increase in $N$ reduces the error by exactly 10×. Even at $N = 100{,}000$ the error is $4.14 \times 10^{-6}$, many orders of magnitude worse than `quad` at the same computational cost. This illustrates why adaptive quadrature is preferred over naive discretisation for smooth integrals.

# Exercise 5.6 — Curve length in a Bernoulli VAE latent space

Train a Bernoulli VAE with a **2-dimensional** latent space on binarised MNIST, then compute the Riemannian length of latent curves using the Fisher–Rao metric of the decoder.

**Fisher–Rao metric for a Bernoulli decoder**

Let $\eta(z)$ be the logits output by the decoder and $\mu(z) = \sigma(\eta(z))$. The metric tensor is:

$$G(z) = J_\eta(z)^\top \operatorname{diag}\!\bigl(\mu(z)\odot(1-\mu(z))\bigr)\, J_\eta(z)$$

where $J_\eta = \partial \eta / \partial z$ is the $D \times M$ Jacobian.

**Curve length (Eq. 4.2)**

$$L(c) = \int_0^1 \sqrt{\dot{c}(t)^\top\, G(c(t))\, \dot{c}(t)}\; dt$$

In [13]:
import sys, os
sys.path.append(os.path.join(os.pardir, "Week1"))

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from helpers import GaussianPrior, GaussianEncoder, BernoulliDecoder, VAE, train

device = "cpu"
M = 2  # latent dimension

# ---------- data (binarised MNIST) ----------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x > 0.5).float().squeeze()),
])
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(os.path.join(os.pardir, "data"), train=True, download=True, transform=transform),
    batch_size=128, shuffle=True,
)

# ---------- model ----------
encoder_net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 512), nn.ReLU(),
    nn.Linear(512, 512), nn.ReLU(),
    nn.Linear(512, M * 2),
)
decoder_net = nn.Sequential(
    nn.Linear(M, 512), nn.ReLU(),
    nn.Linear(512, 512), nn.ReLU(),
    nn.Linear(512, 784),
    nn.Unflatten(-1, (28, 28)),
)

model = VAE(GaussianPrior(M), BernoulliDecoder(decoder_net), GaussianEncoder(encoder_net)).to(device)

# ---------- train (or load) ----------
model_path = os.path.join(os.pardir, "outputs", "models", "bernoulli_vae_2d.pt")
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Loaded model from {model_path}")
else:
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    train(model, optimizer, train_loader, epochs=20, device=device)
    os.makedirs(os.path.dirname(model_path), exist_ok=True)
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to {model_path}")

model.eval()
print("Model ready.")

Loaded model from ../outputs/models/bernoulli_vae_2d.pt
Model ready.


**Result:** The VAE was trained for 20 epochs on binarised MNIST using the ELBO objective, reaching a final loss of $\approx 130.57$ nats/image. The trained weights are saved to disk and reloaded on subsequent runs. The 2-D latent space ($M=2$) is small enough to visualise and integrate over efficiently.

## Part 1 — Riemannian metric tensor and curve-length function

We compute $G(z)$ via forward-mode AD (`jvp`), which needs only $M = 2$ forward passes through the decoder (much cheaper than $D = 784$ backward passes).

In [14]:
from scipy import integrate as sp_integrate


def decoder_logits(model, z):
    """Return the flat logit vector η(z) of the Bernoulli decoder (shape (D,))."""
    return model.decoder.decoder_net(z.unsqueeze(0)).reshape(-1)


@torch.no_grad()
def riemannian_metric(model, z):
    """
    Fisher–Rao metric tensor G(z) for a Bernoulli VAE at a single latent point.

        G(z) = Jη^T  diag(μ(1−μ))  Jη        (M × M)

    Uses forward-mode AD (jvp) → only M forward passes.
    """
    M = z.shape[0]

    def logits_fn(z_in):
        return decoder_logits(model, z_in)

    # Build Jacobian column-by-column via jvp
    J_cols = []
    for j in range(M):
        e_j = torch.zeros(M)
        e_j[j] = 1.0
        _, col = torch.autograd.functional.jvp(logits_fn, (z,), (e_j,))
        J_cols.append(col)
    J = torch.stack(J_cols, dim=1)  # (D, M)

    # Weights
    logits = logits_fn(z)
    mu = torch.sigmoid(logits)
    w = mu * (1.0 - mu)  # (D,)

    # G = J^T diag(w) J
    Jw = J * w.unsqueeze(1)  # (D, M)
    G = Jw.T @ J             # (M, M)
    return G


def curve_length(model, c, t_start=0.0, t_end=1.0, eps=1e-5):
    """
    Riemannian length of an arbitrary latent curve c(t) (Eq. 4.2).

    Parameters
    ----------
    model : VAE with BernoulliDecoder
    c     : callable  t (float) → torch.Tensor of shape (M,)
    t_start, t_end : integration bounds
    eps   : step size for finite-difference derivative of c

    Returns
    -------
    length : float
    error  : float  (quadrature error estimate)
    """
    def integrand(t):
        t_t = torch.tensor(t, dtype=torch.float32)
        z = c(t_t)
        # Finite-difference ċ(t)
        c_dot = (c(t_t + eps) - c(t_t - eps)) / (2.0 * eps)
        G = riemannian_metric(model, z)
        return torch.sqrt(c_dot @ G @ c_dot).item()

    length, error = sp_integrate.quad(integrand, t_start, t_end, limit=200)
    return length, error


print("curve_length() ready — accepts any callable curve c(t).")

curve_length() ready — accepts any callable curve c(t).


## Demo — Length of a second-order polynomial curve

As an example, take the quadratic curve

$$c(t) = \bigl(a_1 t^2 + b_1 t + d_1,\; a_2 t^2 + b_2 t + d_2\bigr), \qquad t \in [0,1]$$

connecting two points in the latent space.

In [16]:
# --- Define a second-order polynomial curve in the 2-D latent space ---
# c(t) = a*t^2 + b*t + d   (a, b, d are 2-D vectors)

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="scipy.integrate")
a = torch.tensor([ 0.5, -0.3])
b = torch.tensor([-1.0,  0.8])
d = torch.tensor([ 0.0,  0.0])   # start point c(0)

def poly_curve(t):
    """Second-order polynomial: c(t) = a*t^2 + b*t + d."""
    return a * t**2 + b * t + d

print(f"c(0) = {poly_curve(torch.tensor(0.0)).tolist()}")
print(f"c(1) = {poly_curve(torch.tensor(1.0)).tolist()}")

# --- Compute Riemannian length ---
L, err = curve_length(model, poly_curve)
print(f"\nRiemannian curve length:  L = {L:.6f}  (±{err:.2e})")

# --- For comparison: Euclidean length of the same curve ---
def euclidean_length(c, t_start=0.0, t_end=1.0, eps=1e-5):
    def integrand(t):
        t_t = torch.tensor(t, dtype=torch.float32)
        c_dot = (c(t_t + eps) - c(t_t - eps)) / (2.0 * eps)
        return torch.linalg.norm(c_dot).item()
    return sp_integrate.quad(integrand, t_start, t_end, limit=200)[0]

L_euclid = euclidean_length(poly_curve)
print(f"Euclidean curve length:   L = {L_euclid:.6f}")
print(f"Ratio (Riemannian / Euclidean): {L / L_euclid:.4f}")

c(0) = [0.0, 0.0]
c(1) = [-0.5, 0.5]


/var/folders/4f/7kcbgj992s5fhk7wcryyv4hc0000gn/T/ipykernel_74651/2558300923.py:67: IntegrationWarning: The maximum number of subdivisions (200) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  length, error = sp_integrate.quad(integrand, t_start, t_end, limit=200)



Riemannian curve length:  L = 31.528033  (±1.59e-02)
Euclidean curve length:   L = 0.715416
Ratio (Riemannian / Euclidean): 44.0695


/var/folders/4f/7kcbgj992s5fhk7wcryyv4hc0000gn/T/ipykernel_74651/3275489555.py:27: IntegrationWarning: The maximum number of subdivisions (200) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  return sp_integrate.quad(integrand, t_start, t_end, limit=200)[0]


**Result:** The quadratic curve travels from $c(0) = (0, 0)$ to $c(1) = (-0.5, 0.5)$ — a Euclidean distance of $\approx 0.715$. Yet the **Riemannian length under the Fisher–Rao metric is $\approx 31.5$**, a ratio of $\sim 44\times$.

This large amplification arises because the Fisher–Rao metric weights each infinitesimal step by $\mu(z)(1-\mu(z))$: pixels near the decision boundary (predicted probability $\approx 0.5$) contribute maximally to the metric. When the curve traverses regions where the decoder is uncertain, the Riemannian length blows up relative to the straight-line Euclidean distance. This is the core motivation for Riemannian geometry in latent spaces — it captures the true "information cost" of moving between representations.